# FPL Statistics Processor
## Notebook to fetch the latest FPL statistics and generate JSON reports

Set `gameweek` to the upcoming gameweek, then run this notebook from top to bottom before asking for squad advice.

The notebook refreshes the selected gameweek's fixtures, backs up completed-gameweek player statistics and your public team picks when missing, and creates a report-ready aggregate using all completed gameweeks.

For example, with `gameweek = 3`, the report input is:

```text
gw/gw3/data/gw_3_aggregated_player_form.json
```

It combines player form from GW1-GW2 with GW3 fixture context. Add your YouTube transcripts to `gw/gw3/transcripts/` before generating a report.

In [1]:
import os

# Ensure all relative paths (gw/, lookup/, reports/) resolve from the project root,
# regardless of where this notebook file lives.
if os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")

print(f"Working directory: {os.getcwd()}")

Working directory: /Users/augustustranter/Documents/Programming/Projects/fpl/fpl_agent


In [2]:
import requests

# Connect to the API
fpl_session = requests.Session()
fpl_base_url = "https://fantasy.premierleague.com/api"

response = fpl_session.get(f"{fpl_base_url}/bootstrap-static/")
response.raise_for_status()

fpl_data = response.json()
print("Connected to the FPL API.")

Connected to the FPL API.


In [3]:
# This variable indicates the gameweek for which we are generating data.
# E.g. if gameweek = 3, it means that gameweeks 1 and 2 have already happened 
# and the gameweek 3 deadline is yet to hit.
gameweek = 4

In [4]:
import json

# Get fixture data for the gameweek
fixtures_url = f"{fpl_base_url}/fixtures/?event={gameweek}"
fixtures_response = fpl_session.get(fixtures_url)
fixtures_response.raise_for_status()

fixtures_data = fixtures_response.json()

folder_path = f"gw/gw{gameweek}/data"
import os
if not os.path.exists(folder_path):
    os.makedirs(folder_path)

# Write fixtures data to the JSON file
file_path = f"{folder_path}/gw_{gameweek}_api_data_fixtures.json"
with open(file_path, 'w') as f:
    json.dump(fixtures_data, f, indent=2)

print(f"Fixtures data written to {file_path}")
print(f"Total fixtures: {len(fixtures_data)}")

Fixtures data written to gw/gw4/data/gw_4_api_data_fixtures.json
Total fixtures: 10


In [5]:
import json
from collections import defaultdict

# Calculate goals scored and conceded across each team's last five completed matches.
team_stats = defaultdict(lambda: {"goals_for": 0, "goals_against": 0})
team_match_count = defaultdict(int)

for fixture in fixtures_data:
    team_stats[fixture["team_h"]]
    team_stats[fixture["team_a"]]

for previous_gameweek in range(gameweek - 1, max(0, gameweek - 6), -1):
    previous_fixtures_path = (
        f"gw/gw{previous_gameweek}/data/"
        f"gw_{previous_gameweek}_api_data_fixtures.json"
    )
    with open(previous_fixtures_path, "r") as f:
        previous_fixtures = json.load(f)

    for fixture in reversed(previous_fixtures):
        if fixture["team_h_score"] is None or fixture["team_a_score"] is None:
            continue

        team_h = fixture["team_h"]
        team_a = fixture["team_a"]

        if team_match_count[team_h] < 5:
            team_stats[team_h]["goals_for"] += fixture["team_h_score"]
            team_stats[team_h]["goals_against"] += fixture["team_a_score"]
            team_match_count[team_h] += 1

        if team_match_count[team_a] < 5:
            team_stats[team_a]["goals_for"] += fixture["team_a_score"]
            team_stats[team_a]["goals_against"] += fixture["team_h_score"]
            team_match_count[team_a] += 1

# Add team form to each target-gameweek fixture.
for fixture in fixtures_data:
    fixture["team_h_stats"] = team_stats[fixture["team_h"]]
    fixture["team_a_stats"] = team_stats[fixture["team_a"]]

file_path = f"{folder_path}/gw_{gameweek}_api_data_fixtures.json"
with open(file_path, "w") as f:
    json.dump(fixtures_data, f, indent=2)

print(f"Fixtures with team form written to {file_path}")
print(f"Matches included per team: {dict(team_match_count)}")

Fixtures with team form written to gw/gw4/data/gw_4_api_data_fixtures.json
Matches included per team: {11: 3, 2: 3, 18: 3, 19: 3, 15: 3, 7: 3, 10: 3, 8: 3, 5: 3, 13: 3, 4: 3, 20: 3, 17: 3, 3: 3, 12: 3, 14: 3, 1: 2, 16: 2, 6: 2, 9: 2}


In [6]:
from collections import defaultdict

# Fetch every fixture (not just this gameweek) so we can assess a run of upcoming
# fixtures per team, not only the immediate next one.
fixture_run_depth = 5
all_fixtures_response = fpl_session.get(f"{fpl_base_url}/fixtures/")
all_fixtures_response.raise_for_status()
all_fixtures_data = all_fixtures_response.json()

fixture_run_by_team = defaultdict(list)
for fixture in all_fixtures_data:
    event = fixture.get("event")
    if event is None or not (gameweek <= event < gameweek + fixture_run_depth):
        continue
    fixture_run_by_team[fixture["team_h"]].append({
        "event": event,
        "opponent": fixture["team_a"],
        "is_home": True,
        "difficulty": fixture["team_h_difficulty"],
    })
    fixture_run_by_team[fixture["team_a"]].append({
        "event": event,
        "opponent": fixture["team_h"],
        "is_home": False,
        "difficulty": fixture["team_a_difficulty"],
    })

for team_fixtures in fixture_run_by_team.values():
    team_fixtures.sort(key=lambda item: item["event"])

print(
    f"Fixture run computed for {len(fixture_run_by_team)} teams across "
    f"GW{gameweek}-GW{gameweek + fixture_run_depth - 1}"
)


Fixture run computed for 20 teams across GW4-GW8


In [7]:
import json
import os

# Extract team and player lookup tables from bootstrap data
teams_lookup = {team['id']: team['name'] for team in fpl_data['teams']}
elements_lookup = {element['id']: {'name': element['first_name'] + ' ' + element['second_name'], 'team': element['team']} for element in fpl_data['elements']}

lookup_folder_path = "lookup"
if not os.path.exists(lookup_folder_path):
    os.makedirs(lookup_folder_path)

# Save team lookup
team_lookup_path = f"{lookup_folder_path}/team_lookup.json"
with open(team_lookup_path, 'w') as f:
    json.dump(teams_lookup, f, indent=2)

# Save element (player) lookup
element_lookup_path = f"{lookup_folder_path}/element_lookup.json"
with open(element_lookup_path, 'w') as f:
    json.dump(elements_lookup, f, indent=2)

print(f"Team lookup saved to {team_lookup_path}")
print(f"Element lookup saved to {element_lookup_path}")
print(f"Total teams: {len(teams_lookup)}")
print(f"Total players: {len(elements_lookup)}")

Team lookup saved to lookup/team_lookup.json
Element lookup saved to lookup/element_lookup.json
Total teams: 20
Total players: 656


In [8]:
# Cache player statistics for every completed gameweek.
for completed_gameweek in range(1, gameweek):
    completed_folder_path = f"gw/gw{completed_gameweek}/data"
    os.makedirs(completed_folder_path, exist_ok=True)
    player_stats_path = (
        f"{completed_folder_path}/"
        f"gw_{completed_gameweek}_api_data_player_stats.json"
    )

    if os.path.exists(player_stats_path):
        with open(player_stats_path, "r") as f:
            cached_player_stats = json.load(f)
        if cached_player_stats:
            print(
                f"Gameweek {completed_gameweek} player stats already exist at "
                f"{player_stats_path}"
            )
            continue

    player_stats_data = {}
    failed_element_ids = []

    for element in fpl_data["elements"]:
        element_id = element["id"]
        try:
            player_url = f"{fpl_base_url}/element-summary/{element_id}/"
            player_response = fpl_session.get(player_url, timeout=30)
            player_response.raise_for_status()
            player_history = player_response.json()

            gameweek_stats = [
                match for match in player_history.get("history", [])
                if match["round"] == completed_gameweek
            ]
            if gameweek_stats:
                player_stats_data[element_id] = {
                    "id": element_id,
                    "name": f"{element['first_name']} {element['second_name']}",
                    "team": element["team"],
                    "position": element["element_type"],
                    **gameweek_stats[0],
                }
        except requests.RequestException:
            failed_element_ids.append(element_id)

    if failed_element_ids:
        raise RuntimeError(
            f"Could not fetch statistics for {len(failed_element_ids)} players in "
            f"gameweek {completed_gameweek}; no file was written."
        )

    with open(player_stats_path, "w") as f:
        json.dump(player_stats_data, f, indent=2)

    print(
        f"Gameweek {completed_gameweek} player stats written to {player_stats_path}"
    )
    print(f"Total players with stats: {len(player_stats_data)}")

Gameweek 1 player stats already exist at gw/gw1/data/gw_1_api_data_player_stats.json
Gameweek 2 player stats already exist at gw/gw2/data/gw_2_api_data_player_stats.json
Gameweek 3 player stats written to gw/gw3/data/gw_3_api_data_player_stats.json
Total players with stats: 654


In [9]:
# Backfill completed gameweek selections for a public FPL team.
my_team_id = 3123442

# Picks are public only after a gameweek has started. This backs up every completed gameweek.
for completed_gameweek in range(1, gameweek):
    picks_url = (
        f"{fpl_base_url}/entry/{my_team_id}/event/{completed_gameweek}/picks/"
    )
    picks_response = fpl_session.get(picks_url, timeout=30)
    picks_response.raise_for_status()

    picks_path = (
        f"gw/gw{completed_gameweek}/data/"
        f"gw_{completed_gameweek}_my_team_picks.json"
    )
    with open(picks_path, "w") as f:
        json.dump(picks_response.json(), f, indent=2)

    print(f"Gameweek {completed_gameweek} picks written to {picks_path}")

Gameweek 1 picks written to gw/gw1/data/gw_1_my_team_picks.json
Gameweek 2 picks written to gw/gw2/data/gw_2_my_team_picks.json
Gameweek 3 picks written to gw/gw3/data/gw_3_my_team_picks.json


In [10]:
from collections import defaultdict

completed_gameweeks = list(range(1, gameweek))

stat_fields = [
    "total_points", "minutes", "starts", "goals_scored", "assists",
    "clean_sheets", "bonus", "expected_goals", "expected_assists",
    "expected_goal_involvements",
]
per_90_fields = ["total_points", "expected_goals", "expected_assists", "expected_goal_involvements"]

# Load every completed gameweek's cached player stats into a per-player match history.
player_matches = defaultdict(list)
for completed_gameweek in completed_gameweeks:
    player_stats_path = (
        f"gw/gw{completed_gameweek}/data/"
        f"gw_{completed_gameweek}_api_data_player_stats.json"
    )
    with open(player_stats_path, "r") as f:
        completed_player_stats = json.load(f)

    for element_id, match in completed_player_stats.items():
        player_matches[int(element_id)].append(match)

# Map each team to its target-gameweek fixture and the opponent's recent goal form.
fixture_by_team = {}
for fixture in fixtures_data:
    team_h = fixture["team_h"]
    team_a = fixture["team_a"]
    fixture_by_team[team_h] = {
        "opponent": team_a,
        "is_home": True,
        "difficulty": fixture["team_h_difficulty"],
        "opponent_form": team_stats[team_a],
    }
    fixture_by_team[team_a] = {
        "opponent": team_h,
        "is_home": False,
        "difficulty": fixture["team_a_difficulty"],
        "opponent_form": team_stats[team_h],
    }


def summarise_form(matches, gameweeks):
    relevant_matches = [match for match in matches if match.get("round") in gameweeks]
    appearances = sum(1 for match in relevant_matches if match.get("minutes", 0) > 0)

    totals = {field: 0.0 for field in stat_fields}
    for match in relevant_matches:
        for field in stat_fields:
            totals[field] += float(match.get(field, 0) or 0)

    averages_per_appearance = {
        field: round(totals[field] / appearances, 2) if appearances else 0.0
        for field in stat_fields
    }
    minutes = totals["minutes"]
    per_90 = {
        field: round(totals[field] / minutes * 90, 2) if minutes else 0.0
        for field in per_90_fields
    }

    return {
        "gameweeks_included": gameweeks,
        "appearances": appearances,
        "totals": totals,
        "averages_per_appearance": averages_per_appearance,
        "per_90": per_90,
    }


print(f"Player matches loaded for {len(player_matches)} players")
print(f"Fixture lookup built for {len(fixture_by_team)} teams")

Player matches loaded for 654 players
Fixture lookup built for 20 teams


In [11]:
aggregated_player_form = {}

for player in fpl_data["elements"]:
    matches = player_matches[player["id"]]
    target_fixture = fixture_by_team.get(player["team"])
    team_fixture_run = fixture_run_by_team.get(player["team"], [])
    recent_form = {}

    for window in (1, 3, 5):
        first_gameweek = max(1, gameweek - window)
        included_gameweeks = list(range(first_gameweek, gameweek))
        window_matches = [
            match for match in matches
            if match.get("round") in included_gameweeks
        ]
        recent_form[f"last_{window}_gameweeks"] = summarise_form(
            window_matches,
            included_gameweeks,
        )

    form_summary = summarise_form(matches, completed_gameweeks)
    average_difficulty = (
        round(sum(item["difficulty"] for item in team_fixture_run) / len(team_fixture_run), 2)
        if team_fixture_run else None
    )
    aggregated_player_form[player["id"]] = {
        "id": player["id"],
        "name": f"{player['first_name']} {player['second_name']}",
        "team": teams_lookup[player["team"]],
        "position": player["element_type"],
        "completed_gameweeks": completed_gameweeks,
        "appearances": form_summary["appearances"],
        "price": player["now_cost"] / 10,
        "ownership_percent": float(player["selected_by_percent"]),
        "transfers_in_event": player["transfers_in_event"],
        "transfers_out_event": player["transfers_out_event"],
        "news": player["news"],
        "chance_of_playing_next_round": player["chance_of_playing_next_round"],
        "totals": form_summary["totals"],
        "averages_per_appearance": form_summary["averages_per_appearance"],
        "per_90": form_summary["per_90"],
        "recent_form": recent_form,
        "upcoming_fixture": {
            "opponent": teams_lookup[target_fixture["opponent"]],
            "is_home": target_fixture["is_home"],
            "difficulty": target_fixture["difficulty"],
            "opponent_form": target_fixture["opponent_form"],
        } if target_fixture else None,
        "fixture_run": {
            "num_fixtures": len(team_fixture_run),
            "average_difficulty": average_difficulty,
            "fixtures": [
                {
                    "gameweek": item["event"],
                    "opponent": teams_lookup[item["opponent"]],
                    "is_home": item["is_home"],
                    "difficulty": item["difficulty"],
                }
                for item in team_fixture_run
            ],
        },
    }

aggregate_path = f"{folder_path}/gw_{gameweek}_aggregated_player_form.json"
with open(aggregate_path, "w") as f:
    json.dump(aggregated_player_form, f, indent=2)

print(f"Aggregated form written to {aggregate_path}")
print(f"Players included: {len(aggregated_player_form)}")


Aggregated form written to gw/gw4/data/gw_4_aggregated_player_form.json
Players included: 656
